In [2]:
from langchain.retrievers import BM25Retriever, EnsembleRetriever 
from langchain_community.vectorstores import FAISS 
from langchain.embeddings import OpenAIEmbeddings 
from langchain_core.documents import Document 


In [3]:
from langchain_core.documents import Document

# Synthetic corpus — small, varied, with a couple of near-duplicates and
# lexical-vs-semantic mismatches so BM25 and dense retrieval disagree.
# Each doc carries metadata you can use later for filtering / inspection.
raw_docs = [
    {
        "id": "doc-01",
        "title": "Resetting a forgotten password",
        "text": (
            "To reset your password, click 'Forgot password' on the login "
            "screen. We email you a reset link that expires after 30 minutes."
        ),
        "category": "account",
    },
    {
        "id": "doc-02",
        "title": "Can't sign in to my account",
        "text": (
            "If you are unable to log in, first confirm your email is verified. "
            "Repeated failed attempts will temporarily lock the account for 15 minutes."
        ),
        "category": "account",
    },
    {
        "id": "doc-03",
        "title": "Updating billing information",
        "text": (
            "Change your card on file under Settings > Billing. Invoices are "
            "generated on the first of each month and charged automatically."
        ),
        "category": "billing",
    },
    {
        "id": "doc-04",
        "title": "Refund policy",
        "text": (
            "Refunds are available within 14 days of purchase. Annual plans are "
            "prorated. Contact support to start a refund request."
        ),
        "category": "billing",
    },
    {
        "id": "doc-05",
        "title": "API rate limits",
        "text": (
            "The REST API allows 100 requests per minute per key. Exceeding the "
            "limit returns HTTP 429. Use exponential backoff to retry."
        ),
        "category": "developer",
    },
    {
        "id": "doc-06",
        "title": "Authenticating API requests",
        "text": (
            "Pass your secret key in the Authorization header as a bearer token. "
            "Never expose keys in client-side code."
        ),
        "category": "developer",
    },
    {
        "id": "doc-07",
        "title": "Exporting your data",
        "text": (
            "You can download a full export of your data as CSV or JSON from "
            "Settings > Privacy. Large exports are emailed as a download link."
        ),
        "category": "data",
    },
    {
        "id": "doc-08",
        "title": "Deleting your account",
        "text": (
            "Account deletion is permanent and removes all stored data after a "
            "30-day grace period. This action cannot be undone."
        ),
        "category": "account",
    },
]

docs = [
    Document(
        page_content=f"{d['title']}\n\n{d['text']}",
        metadata={"id": d["id"], "title": d["title"], "category": d["category"]},
    )
    for d in raw_docs
]

print(f"{len(docs)} documents")
docs[0]


8 documents


Document(metadata={'id': 'doc-01', 'title': 'Resetting a forgotten password', 'category': 'account'}, page_content="Resetting a forgotten password\n\nTo reset your password, click 'Forgot password' on the login screen. We email you a reset link that expires after 30 minutes.")

In [ ]:
bm25 = BM25Retriever.from_documents(docs) 
vector_db = FAISS.from_documents(docs, OpenAIEmbeddings(api_key="")) 
vector_retriever = vector_db.as_retriever() 

# RRF is internal 
hybrid = EnsembleRetriever(
    retrievers=[bm25, vector_retriever], 
    weights=[0.4, 0.6], 
)
query = "I locked myself out and can't get into my account" 
results = hybrid.get_relevant_documents(query) 
for doc in results: 
    print(doc.page_content[:200])

D:\TEMP\ipykernel_29188\102065675.py:11: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  results = hybrid.get_relevant_documents(query)


Can't sign in to my account

If you are unable to log in, first confirm your email is verified. Repeated failed attempts will temporarily lock the account for 15 minutes.
Deleting your account

Account deletion is permanent and removes all stored data after a 30-day grace period. This action cannot be undone.
Resetting a forgotten password

To reset your password, click 'Forgot password' on the login screen. We email you a reset link that expires after 30 minutes.
API rate limits

The REST API allows 100 requests per minute per key. Exceeding the limit returns HTTP 429. Use exponential backoff to retry.
Updating billing information

Change your card on file under Settings > Billing. Invoices are generated on the first of each month and charged automatically.
Exporting your data

You can download a full export of your data as CSV or JSON from Settings > Privacy. Large exports are emailed as a download link.


In [ ]:
# manual 
bm25 = BM25Retriever.from_documents(docs) 
bm25.k = 10 # candidates 

vector_db = FAISS.from_documents(docs, OpenAIEmbeddings(api_key="")) 
vector_retriever = vector_db.as_retriever(search_kwargs={"k": 10}) 
query = "I locked myself out and can't get into my account" 

# gets 2 ranked lists: bm25 sparse search and embedding dense search, fuse with rrf 
bm25_ranked = bm25.invoke(query) # list[Document] 
vector_ranked = vector_retriever.invoke(query) 

def rrf(ranked_lists, k: int = 60, weights=None):
    weights = weights or [1] * len(ranked_lists)
    scores = {}
    docs_by_id = {}
    for ranked, w in zip(ranked_lists, weights):
        for rank, doc in enumerate(ranked):
            doc_id = doc.metadata["id"]          # hashable key
            docs_by_id[doc_id] = doc             # remember the Document
            scores[doc_id] = scores.get(doc_id, 0) + w * 1 / (k + rank)
    ranked_ids = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)
    return [(docs_by_id[doc_id], score) for doc_id, score in ranked_ids]

for doc, score in rrf([bm25_ranked, vector_ranked]):
    print(f"{score:.5f}  {doc.metadata['id']}  {doc.page_content[:80]}")

0.03333  doc-02  Can't sign in to my account

If you are unable to log in, first confirm your ema
0.03252  doc-08  Deleting your account

Account deletion is permanent and removes all stored data
0.03150  doc-05  API rate limits

The REST API allows 100 requests per minute per key. Exceeding 
0.03132  doc-01  Resetting a forgotten password

To reset your password, click 'Forgot password' 
0.03128  doc-03  Updating billing information

Change your card on file under Settings > Billing.
0.03126  doc-07  Exporting your data

You can download a full export of your data as CSV or JSON 
0.03078  doc-04  Refund policy

Refunds are available within 14 days of purchase. Annual plans ar
0.03031  doc-06  Authenticating API requests

Pass your secret key in the Authorization header as
